# Assignment 3: Milestone II — Web-based Data Application


## Introduction

In this notebook, we address the problem of transforming raw cosmetics review text into a clean, standardised vocabulary suitable for downstream machine learning tasks. Working with a corpus of approximately 61,000 reviews from the `cosmetics_beauty_products_reviews.csv` dataset, we implement a systematic preprocessing pipeline that:

- Detects and preserves domain-specific bigram collocations (e.g., _"dry skin"_, _"tea tree"_) before tokenization to avoid losing semantic value.
- Applies the required tokenization, case normalisation, short-word removal, stopword removal, and frequency-based filtering steps.
- Extends the pipeline with **lemmatization** to reduce vocabulary sparsity by collapsing inflected word forms.
- Verifies data integrity throughout with explicit sanity checks at each stage.

**Required outputs:**

- `processed.csv` — the original dataset augmented with a `processed_review_text` column.
- `vocab.txt` — the final vocabulary in `word:index` format, sorted alphabetically.

## Task 1: Functionality for Online Shopper Item Search

### 1.1 Design Rationale

The task specification requires the system to handle three distinct query patterns:

| Pattern | Example | Challenge |
|---|---|---|
| Brand name | `"Maybelline"` | Must match even with typos |
| Category / keyword | `"moisturiser for dry skin"` | No brand involved |
| Brand + category | `"L'Oreal serum"` | Must align both signals |

A single string-matching strategy handles the first two well but collapses on
semantic paraphrasing (e.g. *moisturiser* vs *hydrating cream*). A pure
embedding approach handles paraphrasing but drifts on exact brand names. We
therefore implement **two complementary strategies** fused at the score level:

```
Query
  ├─► Strategy 1: Weighted String Match  ──────────────────────┐
  │      (word coverage + brand alignment + fuzzy fallback)     ├─► max-fusion ─► ranked list
  └─► Strategy 2: GloVe Semantic Search  ──────────────────────┘
         (mean query vector vs. pre-computed product vectors)
```


### 1.2 Case Normalisation

All string comparisons are performed in lower-case. Both the query and every
product title / brand name are passed through `.lower()` before any matching
occurs, so `"MAYBELLINE"`, `"Maybelline"`, and `"maybelline"` are treated
identically.

Beyond case folding, the strategy also handles **typographical errors** via
`difflib.SequenceMatcher`. When no direct word overlap is found, the first
token of the query is compared against the first token of each brand name with
a fuzzy ratio. A query like `"maybeline foundation"` (one *l* missing) scores
≈ 0.94 against `"maybelline"`, comfortably above the 0.7 threshold, and the
product still surfaces in results.


### 1.3 Strategy 1 — Weighted String Matching (with Inverted Index)

String matching is the primary strategy for brand and keyword queries. It
produces a normalised score in **[0, 1]** composed of three weighted sub-signals:

| Sub-signal | Weight | Description |
|---|---|---|
| `match_ratio` | 0.50 | Fraction of query words found in title *or* brand |
| `category_match_ratio` | 0.40 | Fraction of non-brand query words found in title |
| `brand_match` | 0.10 | Binary flag — any query word appears in brand name |

**Inverted Index:** `build_inverted_index(products_df)` constructs a
`word → set[product_id]` map once at startup. At query time, only the *candidate*
product IDs (union of the sets for each query word) are scored — **O(k)** instead
of **O(n)**. When the index returns zero hits (e.g. a misspelled brand name), the
full fuzzy-brand scan fires as a fallback so no results are silently dropped.


In [ ]:
# core/search_engine.py — Strategy 1 with Inverted Index
from difflib import SequenceMatcher
import pandas as pd
from typing import Dict, Optional

def build_inverted_index(products_df: pd.DataFrame) -> Dict[str, set]:
    """Build word → set[product_id] index at startup (called once)."""
    index: Dict[str, set] = {}
    for _, row in products_df.iterrows():
        pid = row['product_id']
        text = (str(row['brand_name']) + ' ' + str(row['product_title'])).lower()
        for word in text.split():
            if len(word) > 1:
                index.setdefault(word, set()).add(pid)
    return index


def normalized_string_match(
    query: str,
    products_df: pd.DataFrame,
    similarity_threshold: float = 0.7,
    inverted_index: Optional[Dict[str, set]] = None,   # NEW
) -> Dict[int, float]:
    query_lower = query.lower()
    query_words = [w for w in query_lower.split() if len(w) > 1]
    if not query_words:
        return {}

    all_brands      = set(products_df['brand_name'].str.lower().unique())
    brand_query_words = [w for w in query_words if w in all_brands]
    results: Dict[int, float] = {}

    def _score_row(row) -> None:
        brand_lower = str(row['brand_name']).lower()
        title_lower = str(row['product_title']).lower()
        matches = [w for w in query_words if w in title_lower or w in brand_lower]
        if not matches:
            brand_fw = brand_lower.split()[0] if brand_lower else ''
            sim = SequenceMatcher(None, query_words[0], brand_fw).ratio()
            if sim >= similarity_threshold:
                results[row['product_id']] = 0.3 * sim
            return
        match_ratio = len(matches) / len(query_words)
        category_words = [w for w in query_words if w not in brand_query_words]
        category_match_ratio = (
            len([w for w in category_words if w in title_lower]) / len(category_words)
            if category_words else 0.0
        )
        brand_match = 1.0 if any(w in brand_lower for w in query_words) else 0.0
        # Score weights: 50% overall match, 40% category match, 10% brand match
        score = 0.5 * match_ratio + 0.4 * category_match_ratio + 0.1 * brand_match
        if len(matches) == len(query_words) and brand_match:
            score = 1.0
        results[row['product_id']] = float(score)

    if inverted_index is not None:
        candidate_ids = set()
        for w in query_words:
            candidate_ids |= inverted_index.get(w, set())
        if candidate_ids:
            for _, row in products_df[products_df['product_id'].isin(candidate_ids)].iterrows():
                _score_row(row)
        else:
            for _, row in products_df.iterrows():   # fuzzy fallback — full scan
                _score_row(row)
    else:
        for _, row in products_df.iterrows():
            _score_row(row)

    return results


### 1.4 Strategy 2 — GloVe Semantic Search (with Query Expansion & OOV Handling)

Strategy 2 handles conceptual paraphrasing by operating in a 300-dimensional
GloVe embedding space.

**Improvement — Query Expansion:** Before embedding, `expand_query(query)` appends
synonyms from two sources:
- A 16-entry **cosmetics domain map** (e.g. `"spf"` → `["sunscreen", "sun protection", ...]`)
  covering terms WordNet handles poorly.
- **NLTK WordNet** noun/adjective synset lemmas (up to 2 synonyms per token).

Expansion applies only to the semantic embedding — string matching still uses the
original query so precision is not diluted.

**OOV-aware weighting:** `get_query_embedding` now returns
`(embedding, oov_ratio)` — the fraction of tokens absent from GloVe. When
`oov_ratio > 0.5` (majority OOV, typically a misspelled brand name), semantic
search is suppressed entirely and string matching carries the result alone.


In [ ]:
# core/search_engine.py — Strategy 2 with Query Expansion & OOV Handling
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from typing import Dict, List, Optional, Tuple

_COSMETICS_MAP = {
    'spf':         ['sunscreen', 'sunblock', 'sun protection', 'uv protection'],
    'moisturiser': ['moisturizer', 'hydrating', 'hydration', 'lotion'],
    'serum':       ['essence', 'treatment', 'ampoule'],
    'foundation':  ['coverage', 'base makeup', 'complexion'],
    # ... (16 entries total in production)
}

def expand_query(query: str) -> str:
    """Append domain synonyms + WordNet lemmas (semantic path only)."""
    from nltk.corpus import wordnet
    tokens, extra = query.lower().split(), []
    for token in tokens:
        if token in _COSMETICS_MAP:
            extra.extend(_COSMETICS_MAP[token])
        else:
            for syn in wordnet.synsets(token, pos=[wordnet.NOUN, wordnet.ADJ])[:2]:
                for lemma in syn.lemmas()[:2]:
                    name = lemma.name().replace('_', ' ')
                    if name != token and len(name) > 2:
                        extra.append(name)
    return (query + ' ' + ' '.join(extra)) if extra else query


def get_query_embedding(
    query: str, embeddings_dict: Dict[str, np.ndarray]
) -> Tuple[Optional[np.ndarray], float]:      # NOW returns (embedding, oov_ratio)
    tokens = query.lower().split()
    if not tokens:
        return None, 1.0
    vecs, oov_count = [], 0
    for w in tokens:
        if w in embeddings_dict:
            vecs.append(embeddings_dict[w])
        else:
            oov_count += 1
    oov_ratio = oov_count / len(tokens)
    return (np.mean(vecs, axis=0), oov_ratio) if vecs else (None, 1.0)


def semantic_search(
    query: str, products_df, product_vectors, embeddings_dict,
    similarity_threshold: float = 0.45,
    _precomputed_embedding: Optional[np.ndarray] = None,  # avoids recomputation
) -> Dict[int, float]:
    embedding = _precomputed_embedding
    if embedding is None:
        embedding, _ = get_query_embedding(query, embeddings_dict)
    if embedding is None:
        return {}
    q_vec = embedding.reshape(1, -1)
    valid_ids = set(products_df['product_id'])
    results = {}
    for pid, pvec in product_vectors.items():
        if pid not in valid_ids:
            continue
        sim = cosine_similarity(q_vec, pvec.reshape(1, -1))[0][0]
        if sim >= similarity_threshold:
            results[pid] = float(sim)   # raw cosine; normalised later in search_products
    return results


### 1.5 Score Fusion (with Min-Max Normalisation)

Results from both strategies are merged using a **max-fusion** rule.

**Score Calibration:** The raw cosine similarity distribution from
Strategy 2 is not on the same scale as the [0, 1] string scores from Strategy 1.
Before fusion, `_minmax_normalize` rescales the entire semantic result set to
**[0, 0.85]**, ensuring the two scales are comparable.

The fusion pipeline in `search_products` is:

```
original query ──► Strategy 1 (string match + inverted index)  ──► string_results [0, 1]

expanded query ──► embedding (oov_ratio) ──► if oov > 50%: skip
               └──► Strategy 2 (semantic)  ──► minmax [0, 0.85]

max-fusion ──► ranked list
```


In [ ]:
# core/search_engine.py — Fusion pipeline with min-max normalisation
from typing import Dict, List

def _minmax_normalize(
    scores: Dict[int, float], lo: float = 0.0, hi: float = 0.85
) -> Dict[int, float]:
    """Rescale a score dict to [lo, hi]; handles single-element sets."""
    if not scores:
        return scores
    vals = list(scores.values())
    min_v, max_v = min(vals), max(vals)
    if max_v == min_v:
        return {pid: hi for pid in scores}
    span = max_v - min_v
    return {pid: lo + (s - min_v) / span * (hi - lo) for pid, s in scores.items()}


def search_products(
    query: str, products_df, product_vectors, embeddings_dict,
    top_n: int = 20,
    inverted_index=None,
) -> List[Dict]:
    if not query or not query.strip():
        return []

    # Expand for semantic embedding; compute once
    expanded = expand_query(query)
    query_embedding, oov_ratio = get_query_embedding(expanded, embeddings_dict)

    # Strategy 1 — original query, fast via inverted index
    string_results = normalized_string_match(query, products_df, inverted_index=inverted_index)

    # Strategy 2 — expanded query, suppressed if oov_ratio > 50 %
    semantic_results: Dict[int, float] = {}
    if oov_ratio <= 0.5 and query_embedding is not None:
        semantic_results = semantic_search(
            expanded, products_df, product_vectors, embeddings_dict,
            _precomputed_embedding=query_embedding,
        )

    # Normalise semantic to [0, 0.85] before fusion
    semantic_results = _minmax_normalize(semantic_results)

    # Max-fusion
    all_scores = {**string_results}
    for pid, score in semantic_results.items():
    # Max-fusion: take the best score from either strategy for each product
        all_scores[pid] = max(all_scores.get(pid, 0.0), score)

    results = []
    for product_id, score in all_scores.items():
        row = products_df[products_df['product_id'] == product_id]
        if row.empty:
            continue
        row = row.iloc[0]
        results.append({'product_id': product_id, 'brand_name': row['brand_name'],
                        'product_title': row['product_title'], 'price': row['price'],
                        'avg_product_rating': row['avg_product_rating'], 'score': score})

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_n]


### 1.6 End-to-End Worked Example

Consider the query **`"maybelline foundation"`**:

**Strategy 1 — String Matching**

| Product | brand_match | match_ratio | category_match_ratio | Score |
|---|---|---|---|---|
| Maybelline Fit Me Foundation | 1.0 | 2/2 = 1.0 | 1/1 = 1.0 | **1.0** (exact) |
| Maybelline SuperStay Blush | 1.0 | 1/2 = 0.5 | 0/1 = 0.0 | 0.35 |
| L'Oreal True Match Foundation | 0.0 | 1/2 = 0.5 | 1/1 = 1.0 | 0.65 |

**Strategy 2 — Semantic Search**

The mean GloVe vector of [`maybelline`, `foundation`] is compared against all
product vectors. Products semantically close to *foundation / makeup / coverage*
score ≥ 0.45 and receive up to 0.9 after the cap.

**After max-fusion**, the Maybelline Fit Me Foundation holds its score of 1.0,
semantically related products that were absent from string results are promoted,
and the final ranked list is returned.

**Typographical robustness:** query `"maybeline foundation"` (one *l* missing)
— `SequenceMatcher("maybeline", "maybelline").ratio()` ≈ 0.94 ≥ 0.7, so the
fuzzy fallback fires and Maybelline products still appear in results.
